In [1]:
# Importing all important packages and opening the zarr file
import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from matplotlib import colors, patches
import matplotlib.image as mpimg
import zarr
import random
import sys
import zipfile
import io
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

colors_THRawS = ['black', 'orange', 'red']
my_cmap_THRawS = ListedColormap(colors_THRawS)

box_0_THRawS = mpatches.Patch(color=colors_THRawS[0], label='0: Certainly not burned area')
box_1_THRawS = mpatches.Patch(color=colors_THRawS[1], label='1: Maybe burned area')
box_2_THRawS = mpatches.Patch(color=colors_THRawS[2], label='2: No burned area')
legend_elements_THRawS = [box_0_THRawS, box_1_THRawS, box_2_THRawS]

In [2]:
# Load data (from 01b_Preprocessing_segTHRawS.ipynb)
bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/segTHRawS'

X_dataset = np.load(os.path.join(bewerkte_data_dir, 'X_train_flat.npy'))
y_dataset = np.load(os.path.join(bewerkte_data_dir, 'y_train_flat.npy'))

# Subsampling (to not crash kernel)
train_set_size = min(100000, int(len(X_dataset) * 0.7))
test_set_size = min(50000, int(len(X_dataset) * 0.3))

X_train, X_test, y_train, y_test = train_test_split(
    X_dataset, 
    y_dataset, 
    train_size=train_set_size, 
    test_size=test_set_size, 
    random_state=42)

all_classes = [0, 1]
names = ["0: Not burned area", "1: Burned area"]

In [ ]:
# ==========================================
# --- RANDOM FOREST MODEL ---
# ==========================================

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)

print(f"\n--- RANDOM FOREST RESULTS ---")
print(f"Test Accuracy: {rf_accuracy*100:.2f}%")
print("Classification Report:")
print(classification_report(y_test, rf_predictions, labels=all_classes, target_names=names, zero_division=0))

np.save(os.path.join(bewerkte_data_dir, 'y_test_masks.npy'), y_test)

rf_probs = rf_model.predict_proba(X_test)[:, 1]
np.save(os.path.join(bewerkte_data_dir, 'y_pred_masks_rf.npy'), rf_predictions)
np.save(os.path.join(bewerkte_data_dir, 'y_pred_probs_fire_rf.npy'), rf_probs)


--- RANDOM FOREST RESULTS ---
Test Accuracy: 99.53%
Classification Report:
                    precision    recall  f1-score   support

0: Not burned area       1.00      1.00      1.00     49428
    1: Burned area       0.87      0.70      0.77       572

          accuracy                           1.00     50000
         macro avg       0.93      0.85      0.89     50000
      weighted avg       1.00      1.00      1.00     50000



In [9]:
# ==========================================
# --- XGBOOST MODEL ---
# ==========================================

dummy_X = np.zeros((4, X_train.shape[1]))
dummy_y = np.array([0, 1, 2, 3])

X_train_xgb = np.vstack((X_train, dummy_X))
y_train_xgb = np.concatenate((y_train, dummy_y))

xgb_model = XGBClassifier(n_estimators=50, max_depth=6, learning_rate=1, n_jobs=-1, random_state=42)
xgb_model.fit(X_train_xgb, y_train_xgb)

xgb_predictions = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_predictions)

print(f"\n--- XGBOOST RESULTS ---")
print(f"Test Accuracy: {xgb_accuracy*100:.2f}%")
print("Classification Report:")
print(classification_report(y_test, xgb_predictions, labels=all_classes, target_names=names, zero_division=0))

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
np.save(os.path.join(bewerkte_data_dir, 'y_pred_masks_xgb.npy'), xgb_predictions)
np.save(os.path.join(bewerkte_data_dir, 'y_pred_probs_fire_xgb.npy'), xgb_probs)


--- XGBOOST RESULTS ---
Test Accuracy: 99.37%
Classification Report:
                    precision    recall  f1-score   support

0: Not burned area       1.00      1.00      1.00     49428
    1: Burned area       0.72      0.74      0.73       572

         micro avg       0.99      0.99      0.99     50000
         macro avg       0.86      0.87      0.86     50000
      weighted avg       0.99      0.99      0.99     50000

